# Unity Catalog Governance — Parker Hannifin Aerospace Component Maintenance

This notebook applies **Unity Catalog governance** across the entire Parker Hannifin predictive maintenance data platform:

| Layer | Tables | Governance Applied |
|-------|--------|--------------------|
| **Bronze** | 5 raw ingestion tables | Table comments, column descriptions, domain tags |
| **Silver** | 1 enriched table | Table comments, column descriptions, quality tags |
| **Gold** | 4 aggregate tables | Table comments, column descriptions, business tags |
| **ML** | 1 prediction table | Table comments, column descriptions, model tags |
| **Metric Views** | 3 semantic views | View comments |

**Catalog**: `genie_zeroops_mfg_catalog`  
**Schema**: `default`  
**Domain**: Aerospace component maintenance, warranty risk, regulatory compliance

In [0]:
%sql
-- ============================================================
-- CATALOG-LEVEL GOVERNANCE
-- ============================================================
ALTER CATALOG genie_zeroops_mfg_catalog
  SET TAGS ('industry' = 'MFG');

COMMENT ON CATALOG genie_zeroops_mfg_catalog IS
  'Parker Hannifin aerospace component predictive maintenance platform. Monitors hydraulic actuators, fuel control valves, pneumatic valves, ECS controllers, and avionics sensors across 10 airline customers and 328 aircraft. Supports proactive maintenance scheduling, warranty risk reduction, and FAA compliance tracking.';

-- ============================================================
-- SCHEMA-LEVEL GOVERNANCE  
-- ============================================================
COMMENT ON SCHEMA genie_zeroops_mfg_catalog.default IS
  'Default schema containing the full medallion architecture (Bronze → Silver → Gold), ML prediction outputs, and semantic metric views for Parker Hannifin aerospace component maintenance analytics.';

In [0]:
%sql
-- ============================================================
-- BRONZE: Customer Registry (Parker's 10 airline customers)
-- ============================================================
COMMENT ON TABLE genie_zeroops_mfg_catalog.default.bronze_customer_registry IS
  'Parker Hannifin airline customer master. Each row represents one airline customer purchasing and operating Parker aerospace components. Source: internal CRM extract.';

ALTER TABLE genie_zeroops_mfg_catalog.default.bronze_customer_registry
  SET TAGS ('quality' = 'bronze');

ALTER TABLE genie_zeroops_mfg_catalog.default.bronze_customer_registry
  ALTER COLUMN airline COMMENT 'Airline customer name (e.g., SkyWest Global, TransPacific Airways)';
ALTER TABLE genie_zeroops_mfg_catalog.default.bronze_customer_registry
  ALTER COLUMN hub COMMENT 'Primary hub airport IATA code for this airline';
ALTER TABLE genie_zeroops_mfg_catalog.default.bronze_customer_registry
  ALTER COLUMN fleet_size COMMENT 'Number of aircraft in the airline fleet using Parker components';

DESCRIBE TABLE EXTENDED genie_zeroops_mfg_catalog.default.bronze_customer_registry;

In [0]:
%sql
-- ============================================================
-- BRONZE: Aircraft Registry (328 aircraft with Parker components)
-- ============================================================
COMMENT ON TABLE genie_zeroops_mfg_catalog.default.bronze_aircraft_registry IS
  'Aircraft fleet registry for all aircraft operating Parker Hannifin components. Contains airframe metadata, age, and cumulative flight hours/cycles. Source: airline fleet management systems.';

ALTER TABLE genie_zeroops_mfg_catalog.default.bronze_aircraft_registry
  SET TAGS ('quality' = 'bronze');

ALTER TABLE genie_zeroops_mfg_catalog.default.bronze_aircraft_registry
  ALTER COLUMN tail_number COMMENT 'Unique aircraft registration (e.g., N101AA). Primary key.';
ALTER TABLE genie_zeroops_mfg_catalog.default.bronze_aircraft_registry
  ALTER COLUMN airline COMMENT 'Operating airline customer name';
ALTER TABLE genie_zeroops_mfg_catalog.default.bronze_aircraft_registry
  ALTER COLUMN aircraft_type COMMENT 'Aircraft model designation (B737-800, A320neo, B787-9, A350-900, E195-E2)';
ALTER TABLE genie_zeroops_mfg_catalog.default.bronze_aircraft_registry
  ALTER COLUMN manufacture_year COMMENT 'Year the airframe was manufactured';
ALTER TABLE genie_zeroops_mfg_catalog.default.bronze_aircraft_registry
  ALTER COLUMN age_years COMMENT 'Aircraft age in years since manufacture';
ALTER TABLE genie_zeroops_mfg_catalog.default.bronze_aircraft_registry
  ALTER COLUMN total_flight_hours COMMENT 'Cumulative airframe flight hours since delivery';
ALTER TABLE genie_zeroops_mfg_catalog.default.bronze_aircraft_registry
  ALTER COLUMN total_cycles COMMENT 'Cumulative pressurization cycles (takeoff-landing pairs)';
ALTER TABLE genie_zeroops_mfg_catalog.default.bronze_aircraft_registry
  ALTER COLUMN base_hub COMMENT 'Home base airport IATA code';

DESCRIBE TABLE EXTENDED genie_zeroops_mfg_catalog.default.bronze_aircraft_registry;

In [0]:
%sql
-- ============================================================
-- BRONZE: Components Registry (1,640 Parker components)
-- ============================================================
COMMENT ON TABLE genie_zeroops_mfg_catalog.default.bronze_components_registry IS
  'Parker Hannifin installed component base. Each row is one Parker component (hydraulic actuator, fuel control valve, pneumatic valve, ECS controller, or avionics sensor) installed on a specific aircraft. Source: Parker aftermarket services database.';

ALTER TABLE genie_zeroops_mfg_catalog.default.bronze_components_registry
  SET TAGS ('quality' = 'bronze');

ALTER TABLE genie_zeroops_mfg_catalog.default.bronze_components_registry
  ALTER COLUMN component_id COMMENT 'Unique Parker component identifier (format: HYD-NXXXAA-001). Primary key.';
ALTER TABLE genie_zeroops_mfg_catalog.default.bronze_components_registry
  ALTER COLUMN part_number COMMENT 'Parker catalog part number (e.g., PH-HYD-7800)';
ALTER TABLE genie_zeroops_mfg_catalog.default.bronze_components_registry
  ALTER COLUMN component_type COMMENT 'Component category: HYDRAULIC_ACTUATOR, FUEL_CONTROL_VALVE, PNEUMATIC_VALVE, ECS_CONTROLLER, AVIONICS_SENSOR';
ALTER TABLE genie_zeroops_mfg_catalog.default.bronze_components_registry
  ALTER COLUMN description COMMENT 'Human-readable component description';
ALTER TABLE genie_zeroops_mfg_catalog.default.bronze_components_registry
  ALTER COLUMN parker_division COMMENT 'Parker Hannifin business division responsible for this component';
ALTER TABLE genie_zeroops_mfg_catalog.default.bronze_components_registry
  ALTER COLUMN tail_number COMMENT 'Aircraft registration where component is installed (FK → bronze_aircraft_registry)';
ALTER TABLE genie_zeroops_mfg_catalog.default.bronze_components_registry
  ALTER COLUMN airline COMMENT 'Operating airline customer';
ALTER TABLE genie_zeroops_mfg_catalog.default.bronze_components_registry
  ALTER COLUMN aircraft_type COMMENT 'Aircraft model designation';
ALTER TABLE genie_zeroops_mfg_catalog.default.bronze_components_registry
  ALTER COLUMN install_date COMMENT 'Date the component was installed on the aircraft (YYYY-MM-DD)';
ALTER TABLE genie_zeroops_mfg_catalog.default.bronze_components_registry
  ALTER COLUMN maintenance_interval_days COMMENT 'Required maintenance interval in days per Parker service bulletin';
ALTER TABLE genie_zeroops_mfg_catalog.default.bronze_components_registry
  ALTER COLUMN max_cycles COMMENT 'Maximum allowable cycles before mandatory overhaul';
ALTER TABLE genie_zeroops_mfg_catalog.default.bronze_components_registry
  ALTER COLUMN cycles_since_install COMMENT 'Accumulated cycles since component installation';
ALTER TABLE genie_zeroops_mfg_catalog.default.bronze_components_registry
  ALTER COLUMN unit_cost_usd COMMENT 'Component unit cost in USD for warranty exposure calculation';
ALTER TABLE genie_zeroops_mfg_catalog.default.bronze_components_registry
  ALTER COLUMN criticality COMMENT 'Component criticality classification: FLIGHT_CRITICAL or ESSENTIAL';

DESCRIBE TABLE EXTENDED genie_zeroops_mfg_catalog.default.bronze_components_registry;

In [0]:
%sql
-- ============================================================
-- BRONZE: Maintenance Records (9,839 service events)
-- ============================================================
COMMENT ON TABLE genie_zeroops_mfg_catalog.default.bronze_maintenance_records IS
  'Component maintenance service history reported by MRO shops and airline maintenance crews. Contains compliance status, anomaly flags, and warranty claim risk indicators. ~9.8% of records have maintenance anomalies. Source: airline maintenance tracking systems and MRO portals.';

ALTER TABLE genie_zeroops_mfg_catalog.default.bronze_maintenance_records
  SET TAGS ('quality' = 'bronze');

ALTER TABLE genie_zeroops_mfg_catalog.default.bronze_maintenance_records
  ALTER COLUMN record_id COMMENT 'Unique maintenance event identifier. Primary key.';
ALTER TABLE genie_zeroops_mfg_catalog.default.bronze_maintenance_records
  ALTER COLUMN component_id COMMENT 'Parker component identifier (FK → bronze_components_registry)';
ALTER TABLE genie_zeroops_mfg_catalog.default.bronze_maintenance_records
  ALTER COLUMN maintenance_date COMMENT 'Date the maintenance was performed (YYYY-MM-DD)';
ALTER TABLE genie_zeroops_mfg_catalog.default.bronze_maintenance_records
  ALTER COLUMN maintenance_type COMMENT 'Service type: SCHEDULED, ON_CONDITION, UNSCHEDULED';
ALTER TABLE genie_zeroops_mfg_catalog.default.bronze_maintenance_records
  ALTER COLUMN mro_shop COMMENT 'MRO facility that performed the maintenance';
ALTER TABLE genie_zeroops_mfg_catalog.default.bronze_maintenance_records
  ALTER COLUMN technician_cert COMMENT 'Technician certification level: A&P, IA, or EASA_B1';
ALTER TABLE genie_zeroops_mfg_catalog.default.bronze_maintenance_records
  ALTER COLUMN compliance_status COMMENT 'Maintenance compliance status: COMPLIANT, OVERDUE, or NON_COMPLIANT';
ALTER TABLE genie_zeroops_mfg_catalog.default.bronze_maintenance_records
  ALTER COLUMN anomaly_flag COMMENT 'True if a maintenance anomaly was detected during this service event';
ALTER TABLE genie_zeroops_mfg_catalog.default.bronze_maintenance_records
  ALTER COLUMN anomaly_type COMMENT 'Anomaly classification: MISSED_MAINTENANCE_INTERVAL, INCORRECT_SERVICE_RECORD, NON_COMPLIANT_PROCEDURE, PREMATURE_WEAR_INDICATOR, UNDOCUMENTED_REPLACEMENT, FALSE_COMPLIANCE_FLAG';
ALTER TABLE genie_zeroops_mfg_catalog.default.bronze_maintenance_records
  ALTER COLUMN anomaly_severity COMMENT 'Anomaly severity tier: LOW, MEDIUM, HIGH, or CRITICAL';
ALTER TABLE genie_zeroops_mfg_catalog.default.bronze_maintenance_records
  ALTER COLUMN days_undetected COMMENT 'Estimated days the anomaly went undetected before discovery';
ALTER TABLE genie_zeroops_mfg_catalog.default.bronze_maintenance_records
  ALTER COLUMN flight_delay_risk COMMENT 'Associated flight delay risk level: LOW, MEDIUM, HIGH, or CRITICAL';
ALTER TABLE genie_zeroops_mfg_catalog.default.bronze_maintenance_records
  ALTER COLUMN warranty_claim_risk COMMENT 'True if this event may trigger a warranty claim against Parker';

DESCRIBE TABLE EXTENDED genie_zeroops_mfg_catalog.default.bronze_maintenance_records;

In [0]:
%sql
-- ============================================================
-- BRONZE: Component Telemetry (1,640 current health readings)
-- ============================================================
COMMENT ON TABLE genie_zeroops_mfg_catalog.default.bronze_component_telemetry IS
  'Current health telemetry snapshot for every installed Parker component. Provides real-time health index, predicted failure timeline, and warranty exposure. One row per component as of the latest reading date. Source: Parker IoT gateway / component health monitoring system.';

ALTER TABLE genie_zeroops_mfg_catalog.default.bronze_component_telemetry
  SET TAGS ('quality' = 'bronze');

ALTER TABLE genie_zeroops_mfg_catalog.default.bronze_component_telemetry
  ALTER COLUMN component_id COMMENT 'Parker component identifier (FK → bronze_components_registry). Primary key.';
ALTER TABLE genie_zeroops_mfg_catalog.default.bronze_component_telemetry
  ALTER COLUMN health_index COMMENT 'Component health score 0.0–1.0 (1.0 = perfect health)';
ALTER TABLE genie_zeroops_mfg_catalog.default.bronze_component_telemetry
  ALTER COLUMN days_to_predicted_failure COMMENT 'ML-estimated days until component failure';
ALTER TABLE genie_zeroops_mfg_catalog.default.bronze_component_telemetry
  ALTER COLUMN maintenance_compliance_rate COMMENT 'Ratio of compliant maintenance events to total events (0.0–1.0)';
ALTER TABLE genie_zeroops_mfg_catalog.default.bronze_component_telemetry
  ALTER COLUMN has_active_anomaly COMMENT 'True if the component currently has an unresolved maintenance anomaly';
ALTER TABLE genie_zeroops_mfg_catalog.default.bronze_component_telemetry
  ALTER COLUMN failure_risk_category COMMENT 'Risk tier: LOW, MEDIUM, HIGH, or CRITICAL';
ALTER TABLE genie_zeroops_mfg_catalog.default.bronze_component_telemetry
  ALTER COLUMN estimated_warranty_exposure_usd COMMENT 'Estimated warranty liability in USD if component fails under current conditions';

DESCRIBE TABLE EXTENDED genie_zeroops_mfg_catalog.default.bronze_component_telemetry;

In [0]:
%sql
-- ============================================================
-- SILVER: Enriched Maintenance (9,839 records)
-- ============================================================
COMMENT ON TABLE genie_zeroops_mfg_catalog.default.silver_enriched_maintenance IS
  'Enriched maintenance records joining bronze service events with component registry and aircraft metadata. Adds compliance scoring, overdue day calculation, anomaly risk flags, and per-record warranty exposure estimates. Used as input to gold aggregations and ML feature engineering.';

ALTER TABLE genie_zeroops_mfg_catalog.default.silver_enriched_maintenance
  SET TAGS ('quality' = 'silver');

ALTER TABLE genie_zeroops_mfg_catalog.default.silver_enriched_maintenance
  ALTER COLUMN compliance_score COMMENT 'Normalized compliance score (0–1) based on overdue days relative to maintenance interval';
ALTER TABLE genie_zeroops_mfg_catalog.default.silver_enriched_maintenance
  ALTER COLUMN overdue_days COMMENT 'Days past the required maintenance interval (0 if on time)';
ALTER TABLE genie_zeroops_mfg_catalog.default.silver_enriched_maintenance
  ALTER COLUMN anomaly_risk_flag COMMENT 'True if anomaly present AND component is FLIGHT_CRITICAL';
ALTER TABLE genie_zeroops_mfg_catalog.default.silver_enriched_maintenance
  ALTER COLUMN record_warranty_exposure_usd COMMENT 'Per-record warranty exposure = unit_cost_usd when warranty risk is true, else 0';

DESCRIBE TABLE EXTENDED genie_zeroops_mfg_catalog.default.silver_enriched_maintenance;

In [0]:
%sql
-- ============================================================
-- GOLD: Component Health (1,640 components)
-- ============================================================
COMMENT ON TABLE genie_zeroops_mfg_catalog.default.gold_component_health IS
  'Component-level health dashboard combining current telemetry readings with aggregated maintenance history. One row per Parker component. Key metrics: health_index, failure risk category, warranty exposure, anomaly counts. Primary table for component health monitoring and proactive maintenance scheduling.';

ALTER TABLE genie_zeroops_mfg_catalog.default.gold_component_health
  SET TAGS ('quality' = 'gold');

ALTER TABLE genie_zeroops_mfg_catalog.default.gold_component_health
  ALTER COLUMN health_index COMMENT 'Real-time component health 0.0–1.0 from telemetry';
ALTER TABLE genie_zeroops_mfg_catalog.default.gold_component_health
  ALTER COLUMN failure_risk_category COMMENT 'ML-derived risk tier: LOW, MEDIUM, HIGH, CRITICAL';
ALTER TABLE genie_zeroops_mfg_catalog.default.gold_component_health
  ALTER COLUMN estimated_warranty_exposure_usd COMMENT 'Projected warranty liability in USD if component fails';
ALTER TABLE genie_zeroops_mfg_catalog.default.gold_component_health
  ALTER COLUMN total_service_events COMMENT 'Count of all maintenance events for this component';
ALTER TABLE genie_zeroops_mfg_catalog.default.gold_component_health
  ALTER COLUMN anomaly_event_count COMMENT 'Count of anomalous maintenance events for this component';
ALTER TABLE genie_zeroops_mfg_catalog.default.gold_component_health
  ALTER COLUMN days_to_predicted_failure COMMENT 'ML-estimated days until predicted component failure';

DESCRIBE TABLE EXTENDED genie_zeroops_mfg_catalog.default.gold_component_health;

In [0]:
%sql
-- ============================================================
-- GOLD: Maintenance Compliance (50 airline×component type groups)
-- ============================================================
COMMENT ON TABLE genie_zeroops_mfg_catalog.default.gold_maintenance_compliance IS
  'Maintenance compliance scorecard aggregated by airline and component type. Tracks compliance rates, anomaly counts, warranty risk events, and SLA breach risk. Used for customer quarterly business reviews and regulatory audit preparation.';

ALTER TABLE genie_zeroops_mfg_catalog.default.gold_maintenance_compliance
  SET TAGS ('quality' = 'gold');

ALTER TABLE genie_zeroops_mfg_catalog.default.gold_maintenance_compliance
  ALTER COLUMN compliance_rate COMMENT 'Fraction of maintenance events that were compliant (0.0–1.0)';
ALTER TABLE genie_zeroops_mfg_catalog.default.gold_maintenance_compliance
  ALTER COLUMN sla_breach_risk COMMENT 'SLA breach risk tier: LOW (<5% non-compliant), MEDIUM (5–15%), HIGH (>15%)';
ALTER TABLE genie_zeroops_mfg_catalog.default.gold_maintenance_compliance
  ALTER COLUMN total_warranty_exposure_usd COMMENT 'Aggregate warranty exposure in USD for this airline×component combination';
ALTER TABLE genie_zeroops_mfg_catalog.default.gold_maintenance_compliance
  ALTER COLUMN avg_overdue_days COMMENT 'Average days past due for overdue maintenance events';

DESCRIBE TABLE EXTENDED genie_zeroops_mfg_catalog.default.gold_maintenance_compliance;

In [0]:
%sql
-- ============================================================
-- GOLD: Maintenance Anomalies (969 anomaly events)
-- ============================================================
COMMENT ON TABLE genie_zeroops_mfg_catalog.default.gold_maintenance_anomalies IS
  'All confirmed maintenance anomaly events enriched with component criticality and Parker division. Each row is a single anomalous maintenance record with severity, delay risk, and warranty impact. Used for anomaly trend analysis, MRO quality auditing, and proactive intervention workflows.';

ALTER TABLE genie_zeroops_mfg_catalog.default.gold_maintenance_anomalies
  SET TAGS ('quality' = 'gold');

ALTER TABLE genie_zeroops_mfg_catalog.default.gold_maintenance_anomalies
  ALTER COLUMN anomaly_type COMMENT '6 categories: MISSED_MAINTENANCE_INTERVAL, INCORRECT_SERVICE_RECORD, NON_COMPLIANT_PROCEDURE, PREMATURE_WEAR_INDICATOR, UNDOCUMENTED_REPLACEMENT, FALSE_COMPLIANCE_FLAG';
ALTER TABLE genie_zeroops_mfg_catalog.default.gold_maintenance_anomalies
  ALTER COLUMN anomaly_severity COMMENT 'Severity tier: LOW, MEDIUM, HIGH, CRITICAL';
ALTER TABLE genie_zeroops_mfg_catalog.default.gold_maintenance_anomalies
  ALTER COLUMN flight_delay_risk COMMENT 'Risk of this anomaly causing flight delays: LOW, MEDIUM, HIGH, CRITICAL';
ALTER TABLE genie_zeroops_mfg_catalog.default.gold_maintenance_anomalies
  ALTER COLUMN days_undetected COMMENT 'Days the anomaly went undetected before discovery';
ALTER TABLE genie_zeroops_mfg_catalog.default.gold_maintenance_anomalies
  ALTER COLUMN record_warranty_exposure_usd COMMENT 'Warranty exposure in USD for this specific anomaly event';

DESCRIBE TABLE EXTENDED genie_zeroops_mfg_catalog.default.gold_maintenance_anomalies;

In [0]:
%sql
-- ============================================================
-- GOLD: Customer Operations (10 airline customers)
-- ============================================================
COMMENT ON TABLE genie_zeroops_mfg_catalog.default.gold_customer_operations IS
  'Per-airline customer operations summary for Parker Hannifin account management. One row per airline with fleet size, compliance rate, anomaly counts, warranty exposure, and top anomaly type. Powers executive dashboards and quarterly business reviews.';

ALTER TABLE genie_zeroops_mfg_catalog.default.gold_customer_operations
  SET TAGS ('quality' = 'gold');

ALTER TABLE genie_zeroops_mfg_catalog.default.gold_customer_operations
  ALTER COLUMN compliance_rate COMMENT 'Overall maintenance compliance rate across all components for this airline (0.0–1.0)';
ALTER TABLE genie_zeroops_mfg_catalog.default.gold_customer_operations
  ALTER COLUMN total_warranty_exposure_usd COMMENT 'Total warranty exposure in USD across all components for this airline';
ALTER TABLE genie_zeroops_mfg_catalog.default.gold_customer_operations
  ALTER COLUMN top_anomaly_type COMMENT 'Most frequent anomaly type detected for this airline';
ALTER TABLE genie_zeroops_mfg_catalog.default.gold_customer_operations
  ALTER COLUMN critical_components COMMENT 'Number of FLIGHT_CRITICAL components serviced by this airline';

DESCRIBE TABLE EXTENDED genie_zeroops_mfg_catalog.default.gold_customer_operations;

In [0]:
%sql
-- ============================================================
-- ML: Part Failure Predictions (1,640 components)
-- ============================================================
COMMENT ON TABLE genie_zeroops_mfg_catalog.default.ml_part_failure_predictions IS
  'ML-generated part failure predictions for every installed Parker component. Combines gradient-boosted binary classifier (AUC 0.999) and Isolation Forest anomaly detection to estimate failure probability, recommended action, and risk category. Refreshed on each model retraining cycle.';

ALTER TABLE genie_zeroops_mfg_catalog.default.ml_part_failure_predictions
  SET TAGS ('quality' = 'gold');

ALTER TABLE genie_zeroops_mfg_catalog.default.ml_part_failure_predictions
  ALTER COLUMN ml_failure_probability COMMENT 'Binary classifier probability of component failure (0.0–1.0)';
ALTER TABLE genie_zeroops_mfg_catalog.default.ml_part_failure_predictions
  ALTER COLUMN recommended_action COMMENT 'ML-recommended maintenance action: ROUTINE_MONITORING, SCHEDULE_INSPECTION, PRIORITY_SERVICE, IMMEDIATE_REPLACEMENT, or EMERGENCY_GROUND';
ALTER TABLE genie_zeroops_mfg_catalog.default.ml_part_failure_predictions
  ALTER COLUMN iso_anomaly COMMENT 'Isolation Forest anomaly flag: 1 = anomalous, 0 = normal';
ALTER TABLE genie_zeroops_mfg_catalog.default.ml_part_failure_predictions
  ALTER COLUMN iso_score COMMENT 'Isolation Forest anomaly score (lower = more anomalous, typical range -0.5 to 0.5)';
ALTER TABLE genie_zeroops_mfg_catalog.default.ml_part_failure_predictions
  ALTER COLUMN failure_risk_category COMMENT 'Risk tier derived from health index + telemetry: LOW, MEDIUM, HIGH, CRITICAL';
ALTER TABLE genie_zeroops_mfg_catalog.default.ml_part_failure_predictions
  ALTER COLUMN estimated_warranty_exposure_usd COMMENT 'Projected warranty cost in USD if this component fails';

DESCRIBE TABLE EXTENDED genie_zeroops_mfg_catalog.default.ml_part_failure_predictions;

In [0]:
%sql
-- ============================================================
-- VALIDATION: Show all governance applied
-- ============================================================

-- 1. Catalog tags
SELECT 'CATALOG' AS scope, 'genie_zeroops_mfg_catalog' AS object_name, tag_name, tag_value
FROM genie_zeroops_mfg_catalog.information_schema.catalog_tags
WHERE catalog_name = 'genie_zeroops_mfg_catalog'

UNION ALL

-- 2. Table tags
SELECT 'TABLE' AS scope, table_name AS object_name, tag_name, tag_value
FROM genie_zeroops_mfg_catalog.information_schema.table_tags
WHERE schema_name = 'default'

ORDER BY scope, object_name, tag_name;

In [0]:
%sql
-- Show table comments across all governed tables
SELECT table_name, comment
FROM genie_zeroops_mfg_catalog.information_schema.tables
WHERE table_schema = 'default'
  AND comment IS NOT NULL
  AND comment != ''
ORDER BY table_name;

In [0]:
%sql
-- Show column-level descriptions for all governed tables
SELECT table_name, column_name, comment
FROM genie_zeroops_mfg_catalog.information_schema.columns
WHERE table_schema = 'default'
  AND comment IS NOT NULL
  AND comment != ''
ORDER BY table_name, ordinal_position;